# DGRN-X-v0.2 Training Pipeline

**Architecture:** DGRN-X-v0.2 (9.68M params) — Serial Conv→Attention + Spatial ValueHead + Auxiliary Heads

**Target:** Google Colab T4 GPU (14GB VRAM)

**Optimizations:**
- Batch 1024 (single batch, no accumulation) — maximize T4 throughput
- Mixed Precision (AMP fp16) — halve memory, 2x compute on Tensor Cores
- cuDNN benchmark — auto-tune conv kernels
- Persistent workers + prefetch — minimize data loading stalls

---

## 1. Setup Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
import os

REPO_ROOT = '/content/drive/MyDrive/chess_engine'
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, 'model'))
sys.path.insert(0, os.path.join(REPO_ROOT, 'train_new_arch'))

# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Copy Data to Local SSD (Avoid I/O Bottleneck)

Đọc mmap trực tiếp từ Google Drive sẽ gây I/O bottleneck nghiêm trọng. Copy data về `/content/local_data` (SSD của Colab) trước khi train.

In [ ]:
import shutil
import time
import os

DRIVE_DATA = '/content/drive/MyDrive/chess_engine/data/process'
LOCAL_DATA = '/content/local_data'

if not os.path.exists(LOCAL_DATA):
    print(f"Copying data from {DRIVE_DATA} to {LOCAL_DATA}...")
    t0 = time.time()
    shutil.copytree(DRIVE_DATA, LOCAL_DATA)
    print(f"Done in {time.time() - t0:.1f}s")
else:
    print(f"Data already exists at {LOCAL_DATA}")

## 3. Configuration

**Chỉnh sửa config bên dưới trước khi train.** Mọi hyperparameter đều ở đây.

In [ ]:
from train_helpers import (
    TrainConfig, set_seed, get_device, build_model, build_dataloaders,
    Trainer, plot_training_history,
    detailed_evaluation, print_detailed_metrics,
)

# ========================= EDIT CONFIG HERE =========================
config = TrainConfig(
    # --- Run ---
    run_name='dgrn_x_v02_run1',

    # --- Paths ---
    data_root='/content/local_data',
    repo_root='/content/drive/MyDrive/chess_engine',
    runs_root='/content/drive/MyDrive/chess_engine/runs',

    # --- Model (match architecture_v3) ---
    input_channels=23,
    width=192,
    board_size=8,
    grid_blocks=12,
    relation_blocks=4,

    # --- Training (optimized for T4 14GB) ---
    epochs=30,
    batch_size=1024,            # T4 handles 1024 easily with AMP
    grad_accum_steps=1,         # no accumulation = fastest throughput
    num_workers=4,

    # --- Optimizer ---
    learning_rate=1e-3,
    min_lr=1e-6,
    weight_decay=1e-4,
    grad_clip_norm=1.0,

    # --- LR Schedule ---
    warmup_epochs=1,
    scheduler_T0=5,             # cosine cycle length (epochs)
    scheduler_T_mult=2,         # cycle grows 2x each restart

    # --- Loss weights ---
    lambda_phase=0.1,
    lambda_material=0.1,

    # --- AMP ---
    use_amp=True,

    # --- Checkpointing ---
    log_every_steps=100,
    eval_every_epoch=1,
    save_every_epoch=1,

    # --- Early stopping (0 = disabled) ---
    patience=10,

    # --- Resume (set path to resume, or None for fresh start) ---
    resume_from=None,
    # resume_from='/content/drive/MyDrive/chess_engine/runs/dgrn_x_v02_run1/checkpoints/best.pt',

    # --- Seed ---
    seed=42,
)

print('Config:')
from dataclasses import asdict
for k, v in asdict(config).items():
    print(f'  {k}: {v}')

## 4. Initialize

In [ ]:
# Set seed + enable cuDNN benchmark
set_seed(config.seed)

# Device
device = get_device()

# Model
model = build_model(config)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nModel: {total_params:,} params ({total_params/1e6:.2f}M)')

# Data
train_loader, val_loader, test_loader = build_dataloaders(config)
print(f'\nTrain batches: {len(train_loader):,}')
print(f'Val batches: {len(val_loader):,}')
if test_loader:
    print(f'Test batches: {len(test_loader):,}')
print(f'\nEffective batch size: {config.batch_size * config.grad_accum_steps}')
print(f'Optimizer steps/epoch: {len(train_loader) // config.grad_accum_steps}')

## 5. Sanity Check

Quick forward/backward pass to verify everything works and measure GPU memory.

In [ ]:
# Sanity check: 1 batch forward + backward
model_test = model.to(device)
model_test.train()

x_test, targets_test = next(iter(train_loader))
x_test = x_test.to(device)
targets_test = targets_test.to(device)
y_v, y_p, y_m = targets_test[:, 0], targets_test[:, 1], targets_test[:, 2]

with torch.amp.autocast(device_type=device.type, enabled=config.use_amp):
    out = model_test(x_test)
    loss_v = torch.nn.functional.mse_loss(out['value'].view(-1), y_v)
    loss_p = torch.nn.functional.mse_loss(out['phase'].view(-1), y_p)
    loss_m = torch.nn.functional.mse_loss(out['material'].view(-1), y_m)
    loss = loss_v + 0.1 * loss_p + 0.1 * loss_m

loss.backward()

print(f'Sanity check PASSED!')
print(f'  Batch size: {x_test.shape[0]}')
print(f'  Value  output: {out["value"].shape}, range [{out["value"].min():.3f}, {out["value"].max():.3f}]')
print(f'  Phase  output: {out["phase"].shape}, range [{out["phase"].min():.3f}, {out["phase"].max():.3f}]')
print(f'  Material output: {out["material"].shape}, range [{out["material"].min():.3f}, {out["material"].max():.3f}]')
print(f'  Loss: {loss.item():.6f} (v={loss_v.item():.4f}, p={loss_p.item():.4f}, m={loss_m.item():.4f})')

if torch.cuda.is_available():
    used = torch.cuda.max_memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'  GPU memory: {used:.2f} / {total:.1f} GB ({used/total*100:.0f}% used)')

model_test.zero_grad()
model_test.cpu()
torch.cuda.empty_cache()

## 6. Train

Checkpoints saved to `runs/<run_name>/checkpoints/`.

To **resume**: set `config.resume_from` to checkpoint path and re-run.

In [ ]:
# Create trainer
trainer = Trainer(
    config=config,
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
)

# Train!
history = trainer.train()

## 7. Visualize Results

In [ ]:
from pathlib import Path

save_path = str(Path(config.runs_root) / config.run_name / 'loss_curves.png')
plot_training_history(history, save_path=save_path)

## 8. Test Evaluation

Evaluate best model on held-out test set.

In [ ]:
if test_loader is not None:
    test_metrics = trainer.evaluate_test(test_loader)
    print(f'\n=== Test Results ===')
    for k, v in test_metrics.items():
        print(f'  {k}: {v:.6f}')
else:
    print('No test set available.')

## 9. Detailed Metrics Analysis

Đánh giá chi tiết model tốt nhất trên **val** và **test** set:
- **MSE / MAE** cho từng head (value, phase, material)
- **Pearson correlation** giữa prediction và ground truth
- **Region-wise MAE**: center (|y|<0.1), mid (0.1≤|y|<0.5), decisive (|y|≥0.5)
- **Prediction statistics**: μ, σ so sánh pred vs target

In [ ]:
import json
from pathlib import Path

# Load best model
best_path = Path(config.runs_root) / config.run_name / 'checkpoints' / 'best.pt'
if best_path.exists():
    ckpt = torch.load(str(best_path), map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    print(f'Loaded best.pt from epoch {ckpt["epoch"]+1}')
else:
    print('No best.pt found, using current model weights')

model = model.to(device)

# === VAL SET ===
print('\n--- Evaluating on VAL set ---')
val_metrics = detailed_evaluation(model, val_loader, device, use_amp=config.use_amp)
print_detailed_metrics(val_metrics, title='Validation Set')

# === TEST SET ===
if test_loader is not None:
    print('--- Evaluating on TEST set ---')
    test_metrics = detailed_evaluation(model, test_loader, device, use_amp=config.use_amp)
    print_detailed_metrics(test_metrics, title='Test Set')
else:
    test_metrics = None
    print('No test set available.')

# Save metrics to JSON
metrics_path = Path(config.runs_root) / config.run_name / 'detailed_metrics.json'
metrics_payload = {'val': val_metrics}
if test_metrics is not None:
    metrics_payload['test'] = test_metrics
with open(str(metrics_path), 'w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, indent=2, ensure_ascii=False)
print(f'Metrics saved to {metrics_path}')

## 10. Resume Training (if needed)

Uncomment and run to resume from a checkpoint.

In [ ]:
# # === RESUME FROM CHECKPOINT ===
# config.resume_from = f'/content/drive/MyDrive/chess_engine/runs/{config.run_name}/checkpoints/best.pt'
# config.epochs = 50  # extend if needed
#
# model = build_model(config)
# train_loader, val_loader, test_loader = build_dataloaders(config)
# trainer = Trainer(config=config, model=model, train_loader=train_loader, val_loader=val_loader, device=device)
# history = trainer.train()

## 11. Standalone Loss Curves Visualization

Mở lại lịch sử huấn luyện từ file `history.json` và vẽ biểu đồ loss (hữu ích khi reload notebook mà không cần chạy lại training).

In [ ]:
import json
from pathlib import Path
from train_helpers import plot_training_history

history_path = Path(config.runs_root) / config.run_name / 'history.json'
if history_path.exists():
    with open(str(history_path), 'r', encoding='utf-8') as f:
        loaded_history = json.load(f)
    print(f"Loaded history from {history_path}")
    plot_training_history(loaded_history, save_path=None)
else:
    print(f"History file not found: {history_path}")